In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

In [15]:
DATA_DIR = Path.cwd().parent / "Datasets"
print(f"Data directory: {DATA_DIR}")

Data directory: c:\Users\laptop\OneDrive\Documents\BankingFinanceAdHocAnalysis\Datasets


In [16]:
account = pd.read_csv(DATA_DIR / "account.csv", sep=None, engine="python")
card = pd.read_csv(DATA_DIR / "card.csv", sep=None, engine="python")
client = pd.read_csv(DATA_DIR / "client.csv", sep=None, engine="python")
disp = pd.read_csv(DATA_DIR / "disp.csv", sep=None, engine="python")
district = pd.read_csv(DATA_DIR / "district.csv", sep=None, engine="python")
loan = pd.read_csv(DATA_DIR / "loan.csv", sep=None, engine="python")
orders = pd.read_csv(DATA_DIR / "order.csv", sep=None, engine="python")

In [17]:
datasets = {
    "ACCOUNT": account,
    "CARD": card,
    "CLIENT": client,
    "DISP": disp,
    "DISTRICT": district,
    "LOAN": loan,
    "ORDERS": orders
}
for name, df in datasets.items():
    print(f"\n{'=' * 40}")
    print(name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column names:", df.columns.tolist())
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


ACCOUNT
Rows: 4500
Columns: 5
Column names: ['account_id', 'district_id', 'frequency', 'date', 'Account_type']
Missing values: 0
Duplicate rows: 0

CARD
Rows: 892
Columns: 4
Column names: ['card_id', 'disp_id', 'type', 'issued']
Missing values: 0
Duplicate rows: 0

CLIENT
Rows: 5369
Columns: 3
Column names: ['client_id', 'birth_number', 'district_id']
Missing values: 0
Duplicate rows: 0

DISP
Rows: 5369
Columns: 4
Column names: ['disp_id', 'client_id', 'account_id', 'type']
Missing values: 0
Duplicate rows: 0

DISTRICT
Rows: 77
Columns: 16
Column names: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']
Missing values: 2
Duplicate rows: 0

LOAN
Rows: 682
Columns: 7
Column names: ['loan_id', 'account_id', 'date', 'amount', 'duration', 'payments', 'status']
Missing values: 0
Duplicate rows: 0

ORDERS
Rows: 6471
Columns: 5
Column names: ['order_id', 'account_id', 'Bank_to', 'account_to', 'amount']
Missing values: 0
Duplicate rows: 0


In [18]:
#head
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(df.head())


ACCOUNT
   account_id  district_id         frequency    date     Account_type
0        1573           63  POPLATEK MESICNE  971229  Savings account
1        3276            1  POPLATEK MESICNE  971229   Salary account
2         124           55  POPLATEK MESICNE  971228   Salary account
3        3958           59  POPLATEK MESICNE  971228      NRI account
4         777           30  POPLATEK MESICNE  971228  Savings account

CARD
   card_id  disp_id     type           issued
0     1005     9285  classic  931107 00:00:00
1      104      588  classic  940119 00:00:00
2      747     4915  classic  940205 00:00:00
3       70      439  classic  940208 00:00:00
4      577     3687  classic  940215 00:00:00

CLIENT
   client_id  birth_number  district_id
0          1        706213           18
1          2        450204            1
2          3        406009            1
3          4        561201            5
4          5        605703            5

DISP
   disp_id  client_id  account_id  

In [19]:
inventory = pd.DataFrame([{"table_name": n, "row_count": len(df), "column_count": len(df.columns), "columns": ", ".join(df.columns)} for n, df in datasets.items()])
display(inventory)

,table_name,row_count,column_count,columns
0,ACCOUNT,4500,5,"account_id, district_id, frequency, date, Acco..."
1,CARD,892,4,"card_id, disp_id, type, issued"
2,CLIENT,5369,3,"client_id, birth_number, district_id"
3,DISP,5369,4,"disp_id, client_id, account_id, type"
4,DISTRICT,77,16,"A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11, ..."
5,LOAN,682,7,"loan_id, account_id, date, amount, duration, p..."
6,ORDERS,6471,5,"order_id, account_id, Bank_to, account_to, amount"


In [20]:
account.head()

,account_id,district_id,frequency,date,Account_type
0,1573,63,POPLATEK MESICNE,971229,Savings account
1,3276,1,POPLATEK MESICNE,971229,Salary account
2,124,55,POPLATEK MESICNE,971228,Salary account
3,3958,59,POPLATEK MESICNE,971228,NRI account
4,777,30,POPLATEK MESICNE,971228,Savings account


In [21]:
print(account['Account_type'].unique())
print(account['Account_type'].value_counts())

['Savings account' 'Salary account' 'NRI account']
Account_type
Savings account    1533
NRI account        1522
Salary account     1445
Name: count, dtype: int64


In [22]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files = ["trnx_16.csv","trnx_17.csv","trnx_18.csv","trnx_19_NEW.csv","trnx_20_NEW.csv","trnx_21_NEW.csv"]
frames = []
for filename in files:
    file_path = DATA_DIR / filename
    df = pd.read_csv(file_path)
    df.columns = (df.columns.str.strip().str.lower().str.replace(" ", "_"))
    df["source_file"] = filename
    frames.append(df)
transactions = pd.concat(frames, ignore_index=True)
transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
output_path = PROCESSED_DIR / "transactions_combined.csv"
transactions.to_csv(output_path, index=False)
print("Consolidation completed!")
print("Rows:", len(transactions))
print("Columns:", len(transactions.columns))
print("Date range:", transactions["date"].min(),
      "to", transactions["date"].max())
print("File saved at:", output_path)

C:\Users\laptop\AppData\Local\Temp\ipykernel_8364\3342620121.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Consolidation completed!
Rows: 1048575
Columns: 11
Date range: 2016-01-01 00:00:00 to 2021-12-19 00:00:00
File saved at: c:\Users\laptop\OneDrive\Documents\BankingFinanceAdHocAnalysis\Datasets\processed\transactions_combined.csv


In [27]:
transactions.head()

,trans_id,account_id,date,type,operation,amount,balance,purpose,bank,account_partern_id,source_file
0,732436,2503,2016-12-31,Withdrawal,Withdrawal in cash,15,27875.5,Payment on Statement,NaN,NaN,trnx_16.csv
1,730715,2497,2016-12-31,Withdrawal,Withdrawal in cash,15,18709.9,Payment on Statement,NaN,NaN,trnx_16.csv
2,733937,2508,2016-12-31,Withdrawal,Withdrawal in cash,15,21579.1,Payment on Statement,NaN,NaN,trnx_16.csv
3,802764,2736,2016-12-31,Withdrawal,Withdrawal in cash,15,32351.4,Payment on Statement,NaN,NaN,trnx_16.csv
4,801171,2732,2016-12-31,Withdrawal,Withdrawal in cash,15,65012.5,Payment on Statement,NaN,NaN,trnx_16.csv


In [28]:
# Check duplicate rows in combined transactions
print("Duplicate rows:", transactions.duplicated().sum())

# Check duplicate transaction IDs
print("Duplicate transaction IDs:", transactions["trans_id"].duplicated().sum())

# Display duplicate rows
duplicates = transactions[transactions.duplicated(keep=False)]
display(duplicates)


Duplicate rows: 0
Duplicate transaction IDs: 0


,trans_id,account_id,date,type,operation,amount,balance,purpose,bank,account_partern_id,source_file


In [29]:
datasets["transactions"] = transactions

print(datasets.keys())

dict_keys(['ACCOUNT', 'CARD', 'CLIENT', 'DISP', 'DISTRICT', 'LOAN', 'ORDERS', 'transactions'])


In [30]:
transactions = pd.read_csv(PROCESSED_DIR / "transactions_combined.csv")

In [33]:
transactions = datasets["transactions"].copy()
transactions["Date_parsed"] = pd.to_datetime(transactions["date"], errors="coerce")

date_report = pd.DataFrame({
    "Metric": ["Total rows", "Missing dates", "Invalid dates", "Earliest date", "Latest date"],
    "Value": [
        len(transactions),
        transactions["date"].isna().sum(),
        (transactions["date"].notna() & transactions["Date_parsed"].isna()).sum(),
        transactions["Date_parsed"].min(),
        transactions["Date_parsed"].max() ]})

display(date_report)


,Metric,Value
0,Total rows,1048575
1,Missing dates,0
2,Invalid dates,0
3,Earliest date,2016-01-01 00:00:00
4,Latest date,2021-12-19 00:00:00


In [ ]:
print("Shape:", transactions.shape)
print("Date range:",transactions["date"].min(),"to",transactions["date"].max())
print("Duplicate transaction IDs:",transactions["trans_id"].duplicated().sum())

Shape: (1048575, 11)
Date range: 2016-01-01 to 2021-12-19
Duplicate transaction IDs: 0


In [34]:
print(orders.shape)
print(orders.columns.tolist())
print(orders.dtypes)
print(orders.head())

(6471, 5)
['order_id', 'account_id', 'Bank_to', 'account_to', 'amount']
order_id        int64
account_id      int64
Bank_to        object
account_to      int64
amount        float64
dtype: object
   order_id  account_id         Bank_to  account_to  amount
0     29401           1        Air Bank    87144583  2452.0
1     29402           2   Bank Creditas    89597016  3372.7
2     29403           2       Fio bank     13943797  7266.0
3     29404           3  Hypotecni bank    83084338  1135.0
4     29405           3       J&T Bank     24485939   327.0


In [35]:
print("CSV rows:", len(transactions))
print("CSV columns:", transactions.shape[1])
print("Date range:", transactions["date"].min(),"to", transactions["date"].max())

CSV rows: 1048575
CSV columns: 12
Date range: 2016-01-01 00:00:00 to 2021-12-19 00:00:00


In [36]:
numeric_columns = ["amount", "balance"]
numeric_results = []
for column in numeric_columns:
    values = pd.to_numeric(transactions[column],errors="coerce")

    numeric_results.append({
        "column": column,
        "missing_values": int(values.isna().sum()),
        "invalid_numeric_values": int(
            (transactions[column].notna() & values.isna()).sum()
        ),
        "negative_values": int((values < 0).sum()),
        "zero_values": int((values == 0).sum()),
        "minimum": values.min(),
        "maximum": values.max()
    })

numeric_report = pd.DataFrame(numeric_results)

display(numeric_report)


,column,missing_values,invalid_numeric_values,negative_values,zero_values,minimum,maximum
0,amount,0,0,0,157,0.0,87400.0
1,balance,0,0,2952,0,-41125.7,209637.0


In [37]:
missing_by_year = (transactions.groupby("source_year")[
        ["bank", "Purpose", "account_partern_id"]].agg([lambda s: s.isna().sum(),lambda s: round(s.isna().mean() * 100, 2)]))

display(missing_by_year)

KeyError: 'source_year'

In [38]:
#  Missing Value Report
missing_report = pd.DataFrame({
    "column": transactions.columns,
    "missing_count": transactions.isna().sum().values,
    "missing_percentage": (transactions.isna().mean().values * 100  ).round(2)})

missing_report = missing_report.sort_values("missing_percentage",ascending=False)

display(missing_report)

,column,missing_count,missing_percentage
9,account_partern_id,753573,71.87
8,bank,627717,59.86
7,purpose,478646,45.65
0,trans_id,0,0.00
3,type,0,0.00
2,date,0,0.00
1,account_id,0,0.00
4,operation,0,0.00
6,balance,0,0.00
5,amount,0,0.00


In [39]:
print(datasets.keys())

dict_keys(['ACCOUNT', 'CARD', 'CLIENT', 'DISP', 'DISTRICT', 'LOAN', 'ORDERS', 'transactions'])


In [40]:
primary_keys = {
    "CLIENT": ["client_id"],
    "ACCOUNT": ["account_id"],
    "DISP": ["disp_id"],
    "CARD": ["card_id"],
    "LOAN": ["loan_id"],
    "ORDERS": ["order_id"],
    "DISTRICT": ["A1"],
    "TRANSACTIONS": ["trans_id"]
}

key_results = []

for table_name, key_columns in primary_keys.items():
    df = datasets[table_name.lower() if table_name == "TRANSACTIONS" else table_name]
    for col in key_columns:
        key_results.append({
            "TABLE": table_name,
            "COLUMN": col,
            "MISSING_KEYS": int(df[col].isna().sum()),
            "DUPLICATE_KEY_ROWS": int(df[col].duplicated(keep=False).sum())
        })

key_report = pd.DataFrame(key_results)
display(key_report)

,TABLE,COLUMN,MISSING_KEYS,DUPLICATE_KEY_ROWS
0,CLIENT,client_id,0,0
1,ACCOUNT,account_id,0,0
2,DISP,disp_id,0,0
3,CARD,card_id,0,0
4,LOAN,loan_id,0,0
5,ORDERS,order_id,0,0
6,DISTRICT,A1,0,0
7,TRANSACTIONS,trans_id,0,0


In [41]:
relationships = [
    ("DISP", "client_id", "CLIENT", "client_id"),
    ("DISP", "account_id", "ACCOUNT", "account_id"),
    ("CARD", "disp_id", "DISP", "disp_id"),
    ("LOAN", "account_id", "ACCOUNT", "account_id"),
    ("ORDERS", "account_id", "ACCOUNT", "account_id"),
    ("CLIENT", "district_id", "DISTRICT", "A1"),
    ("ACCOUNT", "district_id", "DISTRICT", "A1"),
    ("transactions", "account_id", "ACCOUNT", "account_id")
]

relationship_results = []

for child, child_col, parent, parent_col in relationships:
    child_values = datasets[child][child_col].dropna()
    parent_values = set(datasets[parent][parent_col].dropna())
    unmatched = ~child_values.isin(parent_values)

    relationship_results.append({
        "CHILD_TABLE": child,
        "CHILD_COLUMN": child_col,
        "PARENT_TABLE": parent,
        "PARENT_COLUMN": parent_col,
        "NON_NULL_CHILD_ROWS": len(child_values),
        "UNMATCHED_ROWS": int(unmatched.sum()),
        "UNMATCHED_DISTINCT_VALUES": int(child_values[unmatched].nunique())
    })

relationship_report = pd.DataFrame(relationship_results)
display(relationship_report)

,CHILD_TABLE,CHILD_COLUMN,PARENT_TABLE,PARENT_COLUMN,NON_NULL_CHILD_ROWS,UNMATCHED_ROWS,UNMATCHED_DISTINCT_VALUES
0,DISP,client_id,CLIENT,client_id,5369,0,0
1,DISP,account_id,ACCOUNT,account_id,5369,0,0
2,CARD,disp_id,DISP,disp_id,892,0,0
3,LOAN,account_id,ACCOUNT,account_id,682,0,0
4,ORDERS,account_id,ACCOUNT,account_id,6471,0,0
5,CLIENT,district_id,DISTRICT,A1,5369,0,0
6,ACCOUNT,district_id,DISTRICT,A1,4500,0,0
7,transactions,account_id,ACCOUNT,account_id,1048575,0,0


In [42]:
transactions.head()

,trans_id,account_id,date,type,operation,amount,balance,purpose,bank,account_partern_id,source_file,Date_parsed
0,732436,2503,2016-12-31,Withdrawal,Withdrawal in cash,15,27875.5,Payment on Statement,NaN,NaN,trnx_16.csv,2016-12-31
1,730715,2497,2016-12-31,Withdrawal,Withdrawal in cash,15,18709.9,Payment on Statement,NaN,NaN,trnx_16.csv,2016-12-31
2,733937,2508,2016-12-31,Withdrawal,Withdrawal in cash,15,21579.1,Payment on Statement,NaN,NaN,trnx_16.csv,2016-12-31
3,802764,2736,2016-12-31,Withdrawal,Withdrawal in cash,15,32351.4,Payment on Statement,NaN,NaN,trnx_16.csv,2016-12-31
4,801171,2732,2016-12-31,Withdrawal,Withdrawal in cash,15,65012.5,Payment on Statement,NaN,NaN,trnx_16.csv,2016-12-31
